In [1]:
# =====================================
# IMPORTS
# =====================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
# =====================================
# LOAD PREPROCESSED DATA
# =====================================

model_data = pd.read_pickle("../data/preprocessed_model_data.pkl")

print("Loaded preprocessed data:")
print(model_data.shape)
print(model_data.columns.tolist())


Loaded preprocessed data:
(45924, 24)
['Confirmation Year', 'Handler Region', 'Product Group', 'Product Type', 'Product Type Code', 'Industry Name', 'Industry Code', 'Service Line Code', 'Service Line Name', 'Service Detail', 'Service Program', 'Service Catalog Category', 'Service Catalog Item Number', 'Service Catalog Segment', 'Service Catalog Sub Category', 'CCN', 'Ship to Customer Region', 'Has Test Task Flag', 'Ship to Account Number', 'Flex Standards', 'Standard Count', 'Flex Project Count', 'Test Count', 'Log_Eng_Hours']


In [3]:
# =====================================
# CELL 3 - CREATE X AND y WITH CONTROLLED ONE-HOT ENCODING
# =====================================

import numpy as np
import pandas as pd

# -----------------------------
# 1. Define X and y
# -----------------------------

X = model_data.drop(columns=["Log_Eng_Hours"]).copy()
y = model_data["Log_Eng_Hours"].copy()

# -----------------------------
# 2. Identify categorical columns
# -----------------------------

cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Categorical columns before grouping:")
for col in cat_cols:
    print(f"{col}: {X[col].nunique()} unique values")

# -----------------------------
# 3. Limit category explosion
# Keep top categories, group the rest as OTHER
# -----------------------------

MAX_CATEGORIES_PER_COLUMN = 50

X_controlled = X.copy()

for col in cat_cols:
    X_controlled[col] = X_controlled[col].astype("string").fillna("UNKNOWN").str.strip()
    X_controlled[col] = X_controlled[col].replace({"": "UNKNOWN", "nan": "UNKNOWN", "None": "UNKNOWN"})
    
    top_categories = X_controlled[col].value_counts().head(MAX_CATEGORIES_PER_COLUMN).index
    
    X_controlled[col] = np.where(
        X_controlled[col].isin(top_categories),
        X_controlled[col],
        "OTHER"
    )

# -----------------------------
# 4. One-hot encode AFTER grouping
# -----------------------------

X_encoded = pd.get_dummies(
    X_controlled,
    columns=cat_cols,
    drop_first=True,
    dtype=int
)

# -----------------------------
# 5. Clean column names
# -----------------------------

X_encoded.columns = (
    X_encoded.columns
    .astype(str)
    .str.replace("[", "_", regex=False)
    .str.replace("]", "_", regex=False)
    .str.replace("<", "_", regex=False)
)

# -----------------------------
# 6. Confirm results
# -----------------------------

print("\nOriginal X shape:", X.shape)
print("Controlled X shape:", X_controlled.shape)
print("Final X_encoded shape:", X_encoded.shape)
print("y shape:", y.shape)

print("\nFinal encoded feature count:", X_encoded.shape[1])



Categorical columns before grouping:
Confirmation Year: 3 unique values
Handler Region: 3 unique values
Product Group: 218 unique values
Product Type: 233 unique values
Product Type Code: 10 unique values
Industry Name: 31 unique values
Industry Code: 31 unique values
Service Line Code: 167 unique values
Service Line Name: 167 unique values
Service Detail: 386 unique values
Service Program: 120 unique values
Service Catalog Category: 73 unique values
Service Catalog Item Number: 404 unique values
Service Catalog Segment: 22 unique values
Service Catalog Sub Category: 348 unique values
CCN: 209 unique values
Ship to Customer Region: 3 unique values
Has Test Task Flag: 2 unique values
Flex Standards: 201 unique values

Original X shape: (45924, 23)
Controlled X shape: (45924, 23)
Final X_encoded shape: (45924, 646)
y shape: (45924,)

Final encoded feature count: 646


In [4]:
# =====================================
# TRAIN / TEST SPLIT
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.30,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (32146, 646)
X_test: (13778, 646)
y_train: (32146,)
y_test: (13778,)


In [5]:
#Evaluation Function

def evaluate_model(model, X_test, y_test):
    y_pred_log = model.predict(X_test)

    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = r2_score(y_test, y_pred_log)

    return mae, rmse, r2


In [6]:
# =====================================
# TRAIN MODELS
# =====================================

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.001, max_iter=10000),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror"
    )
}

results_list = []

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)

    mae, rmse, r2 = evaluate_model(model, X_test, y_test)

    results_list.append({
        "Model": name,
        "MAE_hours": mae,
        "RMSE_hours": rmse,
        "R2_log_scale": r2
    })

results = pd.DataFrame(results_list).sort_values("RMSE_hours")

results


Training Linear Regression...
Training Ridge Regression...
Training Lasso Regression...
Training Random Forest...
Training XGBoost...


,Model,MAE_hours,RMSE_hours,R2_log_scale
3,Random Forest,6.072709,15.048624,0.534612
4,XGBoost,6.429678,15.919583,0.498288
0,Linear Regression,7.059598,16.885908,0.402516
1,Ridge Regression,7.055704,16.941886,0.404463
2,Lasso Regression,7.277986,17.286118,0.371373


In [7]:
# =====================================
# RANDOM FOREST ONLY - 5 FOLD CV (MAE)
# =====================================

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, make_scorer
import numpy as np

# 5-fold setup
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# MAE scorer on real hours (not log scale)
def mae_real_hours(y_true_log, y_pred_log):
    y_true_hours = np.expm1(y_true_log)
    y_pred_hours = np.expm1(y_pred_log)

    return mean_absolute_error(
        y_true_hours,
        y_pred_hours
    )

mae_scorer = make_scorer(
    mae_real_hours,
    greater_is_better=False
)

# Get Random Forest from your models dictionary
rf = models["Random Forest"]

print("Running 5-Fold CV for Random Forest...")

scores = cross_val_score(
    rf,
    X_encoded,
    y,
    cv=kf,
    scoring=mae_scorer,
    n_jobs=1
)

mae_scores = -scores

print("\nFold MAEs:")
for i, score in enumerate(mae_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print("\nAverage MAE:", round(mae_scores.mean(), 4))
print("MAE Std Dev:", round(mae_scores.std(), 4))



Running 5-Fold CV for Random Forest...

Fold MAEs:
Fold 1: 6.0500
Fold 2: 6.1687
Fold 3: 6.1113
Fold 4: 5.9923
Fold 5: 5.9899

Average MAE: 6.0625
MAE Std Dev: 0.0693


In [8]:
# ============================================================
# DEVIATION BUCKET SUMMARY TABLES
# 100% Data, 70% Training Data, 30% Test Data
# ============================================================

import pandas as pd
import numpy as np

# Use best model
best_model = models["Random Forest"]

# Helper function to create bucket summary
def create_deviation_summary(X_data, y_data, label):
    # Predict log hours
    y_pred_log = best_model.predict(X_data)

    # Convert log hours back to real hours
    actual_hours = np.expm1(y_data)
    predicted_hours = np.expm1(y_pred_log)

    # Create results table
    temp = pd.DataFrame({
        "Actual Human": actual_hours,
        "Model Estimate": predicted_hours
    })

    # Difference between actual and predicted
    temp["Deviation"] = abs(temp["Actual Human"] - temp["Model Estimate"])

    # Buckets
    temp["Deviation Buckets"] = pd.cut(
        temp["Deviation"],
        bins=[-0.001, 1, 2, 3, 7, np.inf],
        labels=["<1 hour", "1-2 hours", "2-3 hours", "3-7 hours", ">7 hours"]
    )

    # Summary
    summary = temp.groupby("Deviation Buckets").agg(
        **{
            "# Projects": ("Deviation", "count"),
            "Actual Human": ("Actual Human", "mean"),
            "Model Estimate Median": ("Model Estimate", "median")
        }
    ).reset_index()

    # Percent of projects
    summary["% of Projects"] = (
        summary["# Projects"] / summary["# Projects"].sum() * 100
    )

    # Reorder columns
    summary = summary[
        ["Deviation Buckets", "# Projects", "% of Projects", "Actual Human", "Model Estimate Median"]
    ]

    # Round
    summary["% of Projects"] = summary["% of Projects"].round(0).astype(int).astype(str) + "%"
    summary["Actual Human"] = summary["Actual Human"].round(2)
    summary["Model Estimate Median"] = summary["Model Estimate Median"].round(2)

    print("\n" + "="*60)
    print(label)
    print("="*60)
    display(summary)

    return summary


# 100% of data
summary_100 = create_deviation_summary(
    X_encoded,
    y,
    "100% of Data"
)

# 70% training data
summary_train = create_deviation_summary(
    X_train,
    y_train,
    "70% Used in Training"
)

# 30% test data
summary_test = create_deviation_summary(
    X_test,
    y_test,
    "30% Not Used in Training"
)



100% of Data


,Deviation Buckets,# Projects,% of Projects,Actual Human,Model Estimate Median
0,<1 hour,18182,40%,5.96,4.37
1,1-2 hours,8680,19%,8.34,6.57
2,2-3 hours,5019,11%,10.59,8.60
3,3-7 hours,7891,17%,14.58,11.36
4,>7 hours,6152,13%,37.29,18.83



70% Used in Training


,Deviation Buckets,# Projects,% of Projects,Actual Human,Model Estimate Median
0,<1 hour,14431,45%,6.10,4.48
1,1-2 hours,6437,20%,8.83,6.98
2,2-3 hours,3382,11%,11.69,9.56
3,3-7 hours,4767,15%,16.93,13.14
4,>7 hours,3129,10%,45.47,23.08



30% Not Used in Training


,Deviation Buckets,# Projects,% of Projects,Actual Human,Model Estimate Median
0,<1 hour,3751,27%,5.43,3.94
1,1-2 hours,2243,16%,6.95,5.47
2,2-3 hours,1637,12%,8.30,6.80
3,3-7 hours,3124,23%,11.00,9.04
4,>7 hours,3023,22%,28.82,15.61


In [9]:
# =====================================
# FEATURE IMPORTANCE FOR BEST MODEL
# =====================================

rf_model = models["Random Forest"]

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(20)


,Feature,Importance
0,Ship to Account Number,0.155024
3,Test Count,0.137302
485,Service Catalog Segment_TST,0.069475
516,Service Catalog Sub Category_Global Market Ser...,0.053390
596,Has Test Task Flag_Yes,0.030686
350,Service Program_New Construction,0.024271
1,Standard Count,0.015745
5,Confirmation Year_2026,0.015638
544,CCN_AAAE,0.014416
6,Handler Region_Americas,0.013775


In [10]:
# =====================================
# FEATURES USED BY FINAL MODEL
# =====================================

rf_model = models["Random Forest"]

print("Number of Features Used:", len(X_train.columns))
print("\nFeatures Used:\n")

for i, col in enumerate(X_train.columns, start=1):
    print(f"{i}. {col}")


Number of Features Used: 646

Features Used:

1. Ship to Account Number
2. Standard Count
3. Flex Project Count
4. Test Count
5. Confirmation Year_2025, 2026
6. Confirmation Year_2026
7. Handler Region_Americas
8. Handler Region_EMEA
9. Product Group_Audio/Video Equipment
10. Product Group_Automotive
11. Product Group_Battery Chargers
12. Product Group_CSP
13. Product Group_Cabinets
14. Product Group_Cameras
15. Product Group_Circuit Breakers
16. Product Group_Computers
17. Product Group_Computing & Peripherals
18. Product Group_Controls
19. Product Group_DG Inverters
20. Product Group_Display & Monitors
21. Product Group_Display Devices
22. Product Group_Displays and Monitors
23. Product Group_Electric Fans - Portable Household Fans
24. Product Group_Electrically Operated Toys
25. Product Group_Entertainment Systems
26. Product Group_Fans
27. Product Group_Gaming Equipment
28. Product Group_Household Cooking Appliances - Cooking Appliance
29. Product Group_Household Cooking Appliances

In [11]:
# =====================================
# ROLL FEATURE IMPORTANCE BACK TO ORIGINAL BUSINESS COLUMNS
# =====================================

rf_model = models["Random Forest"]

encoded_importance = pd.DataFrame({
    "Encoded Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

original_cols = X_controlled.columns.tolist()

def map_to_original_col(encoded_feature):
    for col in sorted(original_cols, key=len, reverse=True):
        if encoded_feature == col or encoded_feature.startswith(col + "_"):
            return col
    return encoded_feature

encoded_importance["Original Business Column"] = encoded_importance["Encoded Feature"].apply(map_to_original_col)

business_importance = (
    encoded_importance
    .groupby("Original Business Column", as_index=False)["Importance"]
    .sum()
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(business_importance.head(15))


,Original Business Column,Importance
0,Ship to Account Number,0.155024
1,Test Count,0.137302
2,Service Catalog Sub Category,0.109295
3,CCN,0.078428
4,Product Type,0.074947
5,Service Catalog Segment,0.072543
6,Service Program,0.064149
7,Product Group,0.050770
8,Service Catalog Item Number,0.032807
9,Service Detail,0.032508


In [12]:
# =====================================
# TOP 10 FEATURE MODEL TEST
# =====================================

top_features = [
    "Ship to Account Number",
    "Test Count",
    "Service Catalog Sub Category",
    "CCN",
    "Product Type",
    "Service Catalog Segment",
    "Service Program",
    "Product Group",
    "Service Catalog Item Number",
    "Service Detail"
]

# Create new X using only top features
X_top = model_data[top_features].copy()
y_top = model_data["Log_Eng_Hours"].copy()

# Find categorical columns
cat_cols_top = X_top.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# Same controlled encoding logic
for col in cat_cols_top:

    X_top[col] = (
        X_top[col]
        .astype("string")
        .fillna("UNKNOWN")
        .str.strip()
    )

    top_categories = (
        X_top[col]
        .value_counts()
        .head(50)
        .index
    )

    X_top[col] = np.where(
        X_top[col].isin(top_categories),
        X_top[col],
        "OTHER"
    )

# One-hot encode
X_top_encoded = pd.get_dummies(
    X_top,
    columns=cat_cols_top,
    drop_first=True,
    dtype=int
)

print("Top 10 model shape:", X_top_encoded.shape)

# Train/test split
X_train_top, X_test_top, y_train_top, y_test_top = train_test_split(
    X_top_encoded,
    y_top,
    test_size=0.2,
    random_state=42
)

# Train RF
rf_top = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf_top.fit(X_train_top, y_train_top)

# Predict
y_pred_log = rf_top.predict(X_test_top)

# Convert back to hours
y_true_hours = np.expm1(y_test_top)
y_pred_hours = np.expm1(y_pred_log)

# MAE
mae_top = mean_absolute_error(
    y_true_hours,
    y_pred_hours
)

print("\nTop 10 Feature Model MAE:", round(mae_top, 2))


Top 10 model shape: (45924, 369)

Top 10 Feature Model MAE: 6.27


In [13]:
# =====================================
# BUCKET ANALYSIS - TOP 10 FEATURE MODEL
# =====================================

import pandas as pd
import numpy as np

# Use predictions from Top 10 model
top10_results = pd.DataFrame({
    "Actual Hours": np.expm1(y_test_top),
    "Predicted Hours": y_pred_hours
})

top10_results["Absolute Error"] = abs(
    top10_results["Actual Hours"] - top10_results["Predicted Hours"]
)

# Create error buckets
top10_results["Error Bucket"] = pd.cut(
    top10_results["Absolute Error"],
    bins=[0, 1, 2, 3, 7, np.inf],
    labels=["<1 hour", "1-2 hours", "2-3 hours", "3-7 hours", ">7 hours"],
    include_lowest=True
)

# Summary table
top10_bucket_summary = (
    top10_results
    .groupby("Error Bucket", observed=False)
    .agg(
        Count=("Absolute Error", "count"),
        Percent=("Absolute Error", lambda x: round(len(x) / len(top10_results) * 100, 2)),
        Actual_Hours_Median=("Actual Hours", "median"),
        Predicted_Hours_Median=("Predicted Hours", "median"),
        Median_Error=("Absolute Error", "median")
    )
    .reset_index()
)

display(top10_bucket_summary)

# Quick business summary
within_1 = top10_results["Absolute Error"].le(1).mean() * 100
within_2 = top10_results["Absolute Error"].le(2).mean() * 100
within_3 = top10_results["Absolute Error"].le(3).mean() * 100
within_7 = top10_results["Absolute Error"].le(7).mean() * 100

print("Top 10 Model Accuracy Buckets:")
print(f"Within 1 hour: {within_1:.2f}%")
print(f"Within 2 hours: {within_2:.2f}%")
print(f"Within 3 hours: {within_3:.2f}%")
print(f"Within 7 hours: {within_7:.2f}%")


,Error Bucket,Count,Percent,Actual_Hours_Median,Predicted_Hours_Median,Median_Error
0,<1 hour,2402,26.15,4.00,3.906855,0.441293
1,1-2 hours,1570,17.09,5.25,5.345656,1.445330
2,2-3 hours,1083,11.79,7.00,6.697094,2.469235
3,3-7 hours,2075,22.59,8.95,8.867494,4.466862
4,>7 hours,2055,22.37,21.80,14.302106,13.175113


Top 10 Model Accuracy Buckets:
Within 1 hour: 26.15%
Within 2 hours: 43.24%
Within 3 hours: 55.04%
Within 7 hours: 77.63%
